# Evaluate

This notebook evaluates the quality of the online alignments in a given experiment directory.

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import os.path
from pathlib import Path
import pandas as pd
import pickle
import re
import import_ipynb
import system_utils
import eval_tools

## Calculate Alignment Errors

First we calculate the alignment errors of a given system on all evaluated measures.

In [ ]:
systems = ['DTW','NOA', 'MATCH']
lags = [0, 100, 200, 300, 400, 500]
scenarios_dir = 'scenarios'

### Evaluate Alignment

In [ ]:
for system_idx in range(len(systems)):
    exp_dir = f'experiments/{systems[system_idx]}'
    eval_dir = f'eval/' + exp_dir[12:] # exp_dir[12:] is the relative path to the experiment directory
    print(f"Evaluating {exp_dir}, saving to {eval_dir}")
    eval_tools.eval_alignment_batch(exp_dir, scenarios_dir, eval_dir)

### Evaluate TSM

In [ ]:
for system_idx in range(len(systems)):
    exp_dir = f'experiments/{systems[system_idx]}'
    eval_dir = f'eval_tsm/' + exp_dir[12:] # exp_dir[12:] is the relative path to the experiment directory
    print(f"Evaluating {exp_dir}, saving to {eval_dir}")
    for lag_idx in range(len(lags)):
        lag = lags[lag_idx]
        eval_dir_with_lag = eval_dir + f'/lag{lag}'
        eval_tools.eval_alignment_batch(exp_dir, scenarios_dir, eval_dir_with_lag, tsm = True, lag = lag)

## Plot Error vs Tolerance

We can visualize the results by plotting the error rate across a range of error tolerances.

In [ ]:
def plotErrorVsTolerance(eval_dirs, maxTol, savefile = None, style='bar', tols = None, bar_tols=None):
    '''
    Plots the error rate across a range of error tolerances.
    
    Inputs
    eval_dir: the eval directories to plot
    maxTol: maximum error tolerance to consider (in milliseconds)
    savefile: if specified, will save the figure to the given filepath
    style: 'bar' or 'line'
    tols: list of error tolerances to evaluate at
    bar_tols: list of error tolerances to plot if using 'bar'
    '''
    
    errRates_list = []
    #color=['blue','orange','green','lightgreen','red'] # hard coded for main results figure
    for i, eval_dir in enumerate(eval_dirs):
    
        # load
        with open(f'{eval_dir}/errs.pkl', 'rb') as f:
            d = pickle.load(f)

        # flattened list
        errs = []
        for scenario_id in d:
            errs = np.append(errs, d[scenario_id])

        # calculate error rates
        if tols is None:
            tols = np.arange(maxTol+1)
        if bar_tols is None:
            bar_tols = tols
        errRates = np.zeros(len(tols))
        for j in range(len(tols)):
            errRates[j] = np.mean(np.abs(errs) > tols[j]/1000)
        errRates_list.append(errRates)
        
        if style == 'line':
            plt.plot(tols, errRates * 100.0)
        elif style == 'bar':
            bar_width = 0.1
            errs = [errRates[tol] * 100.0 for tol in range(len(bar_tols))]
            pos = np.arange(len(errs)) + i * bar_width
            # hard code for main results figure
            #plt.bar(pos, errs, width=bar_width, label=os.path.basename(eval_dir), color = color[i])
            plt.bar(pos, errs, width=bar_width, label=os.path.basename(eval_dir))
            plt.xticks([r + bar_width*len(eval_dirs)/2 for r in range(len(bar_tols))], map(str, bar_tols))
        
    plt.ylabel('Error Rate (%)')
    plt.xlabel('Error Tolerance (ms)')
    plt.legend([os.path.basename(eval_dir) for eval_dir in eval_dirs])
    plt.grid(linestyle='--')
    plt.show()
    if savefile:
        plt.savefig(savefile)

    return errRates_list, tols

Plot the error rate vs error tolerance curve for one system of interest:

In [ ]:
maxTol = 5000 # in milliseconds
tsm = False
tols = [100,200,500, 1000, 2000, 5000]
eval_dir = f'eval{"_tsm" if tsm else ""}/{systems[0]}'
print(eval_dir)
errRates_list, tols = plotErrorVsTolerance([eval_dir], maxTol, tols = tols, savefile=False)
for i in range(len(tols)):
    print(errRates_list[0][i]*100.0)

Overlay multiple error curves for comparison:

In [ ]:
systems_to_compare = systems
tsm = True
eval_dirs = [f'eval/{s}' for s in systems_to_compare]
errRates_list, tols = plotErrorVsTolerance(eval_dirs, maxTol = 1000,tols = tols)

### Plot Lags

In [ ]:
from collections import defaultdict
def aggErrorRates(eval_root, tols = None):
    '''
    Plots the error rate across a range of error tolerances.

    Inputs
    eval_root: the root directory of the eval directories to aggregate
    tols: list of error tolerances to evaluate at
    '''

    errRates_dict = defaultdict(lambda: defaultdict(dict))
    for system_idx in range(len(systems)):
        for lag_idx in range(len(lags)):
            lag = lags[lag_idx]
            eval_dir = f'{eval_root}/{systems[system_idx]}/lag{lag}'

            # load
            with open(f'{eval_dir}/errs.pkl', 'rb') as f:
                d = pickle.load(f)

            # flattened list
            errs = []
            for scenario_id in d:
                errs = np.append(errs, d[scenario_id])

            # calculate error rates
            errRates = {}
            for j in range(len(tols)):
                errRates[tols[j]] = np.mean(np.abs(errs) > tols[j]/1000)
            errRates_dict[systems[system_idx]][lag] = errRates

    return errRates_dict

In [ ]:
eval_root = 'eval_tsm'  
errors_dict = dict(aggErrorRates(eval_root, tols = [100,200,500]))

In [ ]:
# Plot results without evaluation modes; plot a single chart with all systems.

fig, ax = plt.subplots(figsize=(12, 8))

x = np.arange(len(lags))
width = 0.25  # Width of each bar group
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green

for i, system in enumerate(systems):
    # Retrieve error rates for this system across lags/tols
    system_data = errors_dict[system]

    # Collect error rates for each lag at each threshold
    lower_error = [system_data[lag][tols[0]] for lag in lags]
    middle_error = [system_data[lag][tols[1]] for lag in lags]
    upper_error = [system_data[lag][tols[2]] for lag in lags]

    # Convert error rates to accuracy (%)
    lower_accuracy = [(1-v) * 100 for v in lower_error]
    middle_accuracy = [(1-v) * 100 for v in middle_error]
    upper_accuracy = [(1-v) * 100 for v in upper_error]

    # Calculate bar positions for this system
    bar_positions = x + i * width

    # Plot the main bars (middle threshold)
    bars = ax.bar(bar_positions, middle_accuracy, width,
                  label=f'{system.upper()}', color=colors[i], alpha=0.8)

    # Prepare error bars: must all be positive for matplotlib!
    yerr_lower = [max(0, middle_accuracy[j] - lower_accuracy[j]) for j in range(len(middle_accuracy))]
    yerr_upper = [max(0, upper_accuracy[j] - middle_accuracy[j]) for j in range(len(middle_accuracy))]

    # Error bars centered on each bar
    ax.errorbar(
        bar_positions,
        middle_accuracy,
        yerr=[yerr_lower, yerr_upper],
        fmt='none',
        color='black',
        capsize=5,
        capthick=2,
        linewidth=1.5,
        alpha=0.5
    )

    # Add value labels on the main bars
    for j, (pos, acc) in enumerate(zip(bar_positions, middle_accuracy)):
        ax.text(pos, acc + 2, f'{acc:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Customize the plot
ax.set_xlabel('Lag (ms)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Accuracy vs Lag', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels([f'{lag}' for lag in lags], rotation=45)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 105)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

fig.suptitle(
    f'Accuracy vs Lag\n'
    f'Bars: {tols[1]}ms threshold, Error bars: range from {tols[0]}ms to {tols[2]}ms threshold',
    fontsize=16, fontweight='bold'
)
fig.tight_layout(rect=[0, 0, 0.92, 0.93])
plt.show()